<a href="https://colab.research.google.com/github/hgshreyas/AI-LAB/blob/main/1BM24CS108_Week8_FOL_FC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re

class Fact:
    def __init__(self, predicate, arguments):
        self.predicate = predicate
        self.arguments = tuple(arguments)

    def __eq__(self, other):
        return isinstance(other, Fact) and self.predicate == other.predicate and self.arguments == other.arguments

    def __hash__(self):
        return hash((self.predicate, self.arguments))

    def __repr__(self):
        return f"{self.predicate}({', '.join(self.arguments)})"


class Rule:
    def __init__(self, premises, conclusion):
        self.premises = premises
        self.conclusion = conclusion

    def __repr__(self):
        premise_str = " AND ".join([f"{p}({', '.join(a)})" for p, a in self.premises])
        return f"IF {premise_str} -> THEN {self.conclusion[0]}({', '.join(self.conclusion[1])})"


def clean_text_to_standard(text):
    """Normalizes logical operators from math symbols or English words to standard tokens."""
    # Convert operators
    text = text.replace("∧", " AND ").replace("^", " AND ").replace("and", " AND ")
    text = text.replace("→", " -> ").replace("->", " -> ").replace("THEN", " -> ").replace("then", " -> ")
    text = text.replace("IF", "").replace("if", "")

    # Standardize spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def parse_flexible_predicate(text, default_arg="x"):
    """
    Parses flexible inputs like 'Fever(x)', 'has a fever', or 'fever'
    into a structured tuple: (PredicateName, [args])
    """
    text = text.strip()

    # Case 1: Standard structured format already present -> e.g., Fever(P) or Fever(x)
    match = re.match(r"([A-Za-z0-9_]+)\s*\(([^)]+)\)", text)
    if match:
        pred = match.group(1)
        args = [a.strip() for a in match.group(2).split(",")]
        return pred, args

    # Case 2: Natural English strings -> e.g., "patient has a fever", "has muscle aches"
    # Remove filler conversational tokens
    clean_text = re.sub(r"\b(a|an|the|patient|has|have|signs|of)\b", "", text, flags=re.IGNORECASE)
    clean_text = "".join(c for c in clean_text if c.isalnum() or c.isspace()).strip()

    # CamelCase conversion from word list
    words = clean_text.split()
    if not words:
        raise ValueError(f"Could not parse valid phrase expression from: '{text}'")

    pred_name = "".join(w.capitalize() for w in words)
    return pred_name, [default_arg]


def match_premise(premise, kb_facts, substitution):
    p_pred, p_args = premise
    matches = []

    for fact in kb_facts:
        if fact.predicate != p_pred or len(fact.arguments) != len(p_args):
            continue

        new_sub = substitution.copy()
        possible_match = True

        for p_arg, f_arg in zip(p_args, fact.arguments):
            if p_arg.islower():  # Variable
                if p_arg in new_sub and new_sub[p_arg] != f_arg:
                    possible_match = False
                    break
                new_sub[p_arg] = f_arg
            else:  # Constant
                if p_arg != f_arg:
                    possible_match = False
                    break

        if possible_match:
            matches.append(new_sub)

    return matches


def match_all_premises(premises, kb_facts, substitution=None):
    if substitution is None:
        substitution = {}
    if not premises:
        return [substitution]

    first, rest = premises[0], premises[1:]
    first_matches = match_premise(first, kb_facts, substitution)

    all_bindings = []
    for sub in first_matches:
        bindings = match_all_premises(rest, kb_facts, sub)
        all_bindings.extend(bindings)

    return all_bindings


def fol_fc_ask(kb_rules, kb_facts, query_predicate):
    print("\n--- Running Forward Chaining Inference Engine ---")

    for fact in kb_facts:
        if fact.predicate.lower() == query_predicate.lower():
            print(f"🎉 Goal directly found: {fact}")
            return {"x": fact.arguments}

    while True:
        new_facts = set()

        for rule in kb_rules:
            substitutions = match_all_premises(rule.premises, kb_facts)

            for theta in substitutions:
                c_pred, c_args = rule.conclusion
                inferred_args = [theta[arg] if arg in theta else arg for arg in c_args]
                inferred_fact = Fact(c_pred, inferred_args)

                if inferred_fact not in kb_facts and inferred_fact not in new_facts:
                    print(f"🔥 Rule triggered! Derived: {inferred_fact}")
                    new_facts.add(inferred_fact)

                    if inferred_fact.predicate.lower() == query_predicate.lower():
                        kb_facts.update(new_facts)
                        print(f"\n🎉 Goal Reached! Confirmed assertion '{query_predicate}'.")
                        return theta

        if not new_facts:
            print("\n❌ End of stream: No additional deductions available.")
            return False

        kb_facts.update(new_facts)


def main():
    print("======================================================")
    print("   Multi-Format First-Order Logic Reasoning Engine   ")
    print("======================================================")
    print("Supports: English phrases, Math logic (^, ∧, ->) & Standard FOL\n")

    # Ask for tracking target ID
    patient_id = input("Enter individual entity label (e.g., P, John): ").strip()
    if not patient_id:
        patient_id = "P"

    # 1. Collect Facts
    print("\n--- STEP 1: Enter Initial Facts ---")
    kb_facts = set()
    while True:
        inp = input("Add Fact: ").strip()
        if inp.lower() == 'done':
            break
        if not inp:
            continue
        try:
            pred, args = parse_flexible_predicate(inp, default_arg=patient_id)
            # If user provided variable string or fallback, lock entity constant
            if args == ["x"]:
                args = [patient_id]
            kb_facts.add(Fact(pred, args))
        except Exception as e:
            print(f"Parsing error: {e}")

    # 2. Collect Rules
    print("\n--- STEP 2: Enter Inference Rules ---")
    print("Type 'done' when finished.")
    kb_rules = []
    while True:
        inp = input("Add Rule: ").strip()
        if inp.lower() == 'done':
            break
        if not inp:
            continue
        try:
            normalized = clean_text_to_standard(inp)
            if "->" not in normalized:
                print("Error: Could not locate directional map operator ('->' or 'THEN')")
                continue

            premise_part, conclusion_part = normalized.split("->")

            premises_raw = premise_part.split(" AND ")
            premises = [parse_flexible_predicate(p, "x") for p in premises_raw if p.strip()]
            conclusion = parse_flexible_predicate(conclusion_part, "x")

            kb_rules.append(Rule(premises, conclusion))
        except Exception as e:
            print(f"Parsing error on rule array: {e}")

    # 3. Collect Target Query
    print("\n--- STEP 3: Define Target Goal Query ---")
    goal_raw = input("What are you trying to prove? (e.g., TestFlu): ").strip()
    goal_pred, _ = parse_flexible_predicate(goal_raw, "x")

    if not kb_facts or not kb_rules or not goal_pred:
        print("\nAborted. Baseline configuration matrix empty.")
        return

    print("\n================ PARSED KNOWLEDGE BASE ================")
    print(f"Compiled Facts Base: {kb_facts}")
    print("Compiled Rules Base:")
    for idx, r in enumerate(kb_rules, 1):
        print(f"  [{idx}] {r}")
    print(f"Goal State Target: Prove {goal_pred}({patient_id})")
    print("=======================================================")

    # Run Algorithm Engine
    result = fol_fc_ask(kb_rules, kb_facts, goal_pred)
    print(f"\nFinal Engine Matrix Solution: {result}")


if __name__ == "__main__":
    main()


   Multi-Format First-Order Logic Reasoning Engine   
Supports: English phrases, Math logic (^, ∧, ->) & Standard FOL

Enter individual entity label (e.g., P, John): Robert

--- STEP 1: Enter Initial Facts ---
Add Fact: American(Robert)
Add Fact: Enemy(A, America)
Add Fact: Owns(A, T1)
Add Fact: Missile(T1)
Add Fact: done

--- STEP 2: Enter Inference Rules ---
Type 'done' when finished.
Add Rule: American(p) ^ Weapon(q) ^ Sells(p, q, r) ^ Hostile(r) -> Criminal(p)
Add Rule: Missile(x) ^ Owns(A, x) -> Sells(Robert, x, A)
Add Rule: Missile(x) -> Weapon(x)
Add Rule: Enemy(x, America) -> Hostile(x)
Add Rule: done

--- STEP 3: Define Target Goal Query ---
What are you trying to prove? (e.g., TestFlu): Criminal

================ PARSED KNOWLEDGE BASE ================
Compiled Facts Base: {American(Robert), Owns(A, T1), Enemy(A, America), Missile(T1)}
Compiled Rules Base:
  [1] IF American(p) AND Weapon(q) AND Sells(p, q, r) AND Hostile(r) -> THEN Criminal(p)
  [2] IF Missile(x) AND Owns(A, x